# 01 — Data Pipeline

This notebook builds the complete data pipeline: environment setup, document scraping, chunking, embedding, and vector store creation.


- **Data source**: Microsoft Learn training module units — curated pages covering Copilot Studio (AI Champion track) and M365 Copilot usage (User track)
- **External API**: Tavily Search for freshness checks on latest Copilot Studio updates
- **Chunking strategy**: `RecursiveCharacterTextSplitter` with chunk_size=1000, overlap=200


We chose `RecursiveCharacterTextSplitter` with chunk_size=1000 and overlap=200 because:
1. Microsoft Learn content is structured with clear headings and paragraphs. The recursive splitter respects these boundaries.
2. 1000 chars (~200-250 tokens) captures a complete concept while keeping retrieval precise
3. 200-char overlap prevents losing context at chunk boundaries.







In [1]:
import os
import time
import json
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "certops_docs"
DOCS_DIR = Path("../data/docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

required_keys = ["OPENAI_API_KEY", "QDRANT_URL", "QDRANT_API_KEY", "TAVILY_API_KEY", "LANGCHAIN_API_KEY"]
for key in required_keys:
    val = os.getenv(key)
    print(f"  {key}: {'loaded' if val else 'MISSING'}")

  OPENAI_API_KEY: loaded
  QDRANT_URL: loaded
  QDRANT_API_KEY: loaded
  TAVILY_API_KEY: loaded
  LANGCHAIN_API_KEY: loaded


## 1. Document Sources

CertOps supports **two certification tracks**, each with its own curated corpus:

| Track | `audience` value | Focus |
|-------|-----------------|-------|
| M365 Copilot User | `user` | Using Copilot in Word, Excel, Teams, Outlook, PowerPoint |
| Copilot Agent Builder | `ai_champion` | Building agents in Copilot Studio |

Sources come from Microsoft Learn training modules.

**Why so many URLs?** Each Microsoft Learn training module page is relatively concise. Content is spread across multiple sub-pages within a module. Since there are no comprehensive handbooks for Microsoft Copilot agents, we scrape from the available documentation and training module units to assemble a rich enough corpus.

Each url is accompanied by key value pairs repesenting its associated certification track and subject area.


In [2]:
AI_CHAMPION_SOURCES = [
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-bots/3-create-bots", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-bots/create-agents", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-bots/4-topics", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-bots/generative-ai", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-bots/5-test-bots", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-bots/6-performance-analysis", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/create-copilots-copilot-studio/copilot-creation-with-natural", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/create-copilots-copilot-studio/copilot-generative-answers", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/create-copilots-copilot-studio/copilot-deploy", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/manage-power-virtual-agents-topics/2-topics", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/manage-power-virtual-agents-topics/3-branch", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/manage-power-virtual-agents-topics/6-fall-back-topics", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-entities/custom-entities", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-entities/use-entities", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/power-virtual-agents-entities/variables", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/enhance-power-virtual-agents-bots/2-power-automate", "domain": "connectors", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/enhance-power-virtual-agents-bots/actions", "domain": "connectors", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/enhance-power-virtual-agents-bots/triggers", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/build-effective-bots/design", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/build-effective-bots/conversational", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/build-effective-bots/analytics", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/autonomous-agent/bring-autonomous-agents-into-enterprise", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/autonomous-agent/test-refine-autonomous-agent", "domain": "architecture", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/microsoft-copilot-studio/security-and-governance", "domain": "security", "audience": "ai_champion"},
    {"url": "https://learn.microsoft.com/en-us/microsoft-copilot-studio/responsible-ai-overview", "domain": "governance", "audience": "ai_champion"},
]

USER_SOURCES = [
    {"url": "https://learn.microsoft.com/en-us/copilot/overview", "domain": "overview", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/copilot/tutorials/learn-microsoft-copilot/03-copilot-scenarios", "domain": "overview", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/copilot/microsoft-365/microsoft-365-copilot-enablement-resources", "domain": "adoption", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/create-draft-content-with-microsoft-copilot-microsoft-365/2-draft-content-microsoft-copilot-word", "domain": "productivity", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/create-draft-content-with-microsoft-copilot-microsoft-365/3-build-new-slides-agendas-to-do-lists-microsoft-copilot-powerpoint", "domain": "productivity", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/create-draft-content-with-microsoft-copilot-microsoft-365/4-draft-emails-replies-meeting-agendas-microsoft-copilot-outlook", "domain": "communication", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/create-draft-content-with-microsoft-copilot-microsoft-365/5-brainstorm-new-ideas-lists-reports-across-microsoft-365-microsoft-copilot", "domain": "prompting", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/edit-transform-content-with-microsoft-copilot-microsoft-365/2-write-organize-transform-content-copilot-word", "domain": "productivity", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/edit-transform-content-with-microsoft-copilot-microsoft-365/3-add-images-slides-organize-presentation-copilot-powerpoint", "domain": "productivity", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/edit-transform-content-with-microsoft-copilot-microsoft-365/4-format-sort-filter-highlight-data-copilot-excel", "domain": "data-analysis", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/edit-transform-content-with-microsoft-copilot-microsoft-365/5-rewrite-messages-replies-tone-copilot-outlook", "domain": "communication", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/ask-analyze-content-with-microsoft-copilot-microsoft-365/2-ask-copilot-recommendations-help-word", "domain": "productivity", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/ask-analyze-content-with-microsoft-copilot-microsoft-365/4-analyze-work-tables-using-copilot-excel", "domain": "data-analysis", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/ask-analyze-content-with-microsoft-copilot-microsoft-365/5-ask-questions-about-notes-copilot-onenote", "domain": "productivity", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/ask-analyze-content-with-microsoft-copilot-microsoft-365/6-chat-copilot-about-meetings-messages-teams-outlook", "domain": "communication", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/summarize-simplify-information-with-microsoft-copilot-microsoft-365/2-simplify-extract-key-information-copilot-word", "domain": "productivity", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/summarize-simplify-information-with-microsoft-copilot-microsoft-365/4-spot-trends-visualize-data-copilot-excel", "domain": "data-analysis", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/summarize-simplify-information-with-microsoft-copilot-microsoft-365/5-highlight-key-decisions-action-items-teams-meetings", "domain": "communication", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/summarize-simplify-information-with-microsoft-copilot-microsoft-365/6-catch-up-prepare-week-copilot-outlook", "domain": "communication", "audience": "user"},
    {"url": "https://learn.microsoft.com/en-us/training/modules/summarize-simplify-information-with-microsoft-copilot-microsoft-365/7-summarize-information-topic-copilot-microsoft-365", "domain": "prompting", "audience": "user"},
]

ALL_SOURCES = AI_CHAMPION_SOURCES + USER_SOURCES
print(f"AI Champion sources: {len(AI_CHAMPION_SOURCES)}")
print(f"User sources: {len(USER_SOURCES)}")
print(f"Total: {len(ALL_SOURCES)}")

AI Champion sources: 25
User sources: 20
Total: 45


## 2. Scrape and Convert to Markdown

Each URL is fetched, the main content is extracted from the HTML, and converted to clean Markdown.
We save each page locally to `data/docs/` for reproducibility.

In [3]:
import httpx
from bs4 import BeautifulSoup
from markdownify import markdownify as md


def scrape_ms_learn_page(url: str) -> str:
    """Fetch a Microsoft Learn page and return clean Markdown content."""
    headers = {"User-Agent": "CertOps-Scraper/1.0 (educational project)"}
    response = httpx.get(url, headers=headers, follow_redirects=True, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    content = soup.find("main") or soup.find("div", {"id": "main-column"})
    if not content:
        content = soup.find("div", class_="content")
    if not content:
        content = soup.body

    for tag in content.find_all(["nav", "header", "footer", "script", "style"]):
        tag.decompose()

    markdown_text = md(str(content), heading_style="ATX", strip=["img"])
    lines = [line.rstrip() for line in markdown_text.splitlines()]
    cleaned = "\n".join(lines)
    while "\n\n\n" in cleaned:
        cleaned = cleaned.replace("\n\n\n", "\n\n")
    return cleaned.strip()


scraped_docs = []
for source in ALL_SOURCES:
    slug = source["url"].split("/")[-1]
    filepath = DOCS_DIR / f"{slug}.md"
    print(f"Scraping: {slug[:50]}...", end=" ")
    try:
        content = scrape_ms_learn_page(source["url"])
        filepath.write_text(content, encoding="utf-8")
        scraped_docs.append({"slug": slug, "content": content, **source})
        print(f"{len(content):,} chars")
    except Exception as e:
        print(f"FAILED: {e}")
    time.sleep(1)

print(f"\nSuccessfully scraped {len(scraped_docs)}/{len(ALL_SOURCES)} documents")

Scraping: 3-create-bots... 5,537 chars
Scraping: create-agents... 4,841 chars
Scraping: 4-topics... 4,431 chars
Scraping: generative-ai... 7,085 chars
Scraping: 5-test-bots... 3,600 chars
Scraping: 6-performance-analysis... 7,964 chars
Scraping: copilot-creation-with-natural... 6,319 chars
Scraping: copilot-generative-answers... 13,105 chars
Scraping: copilot-deploy... 3,935 chars
Scraping: 2-topics... 21,339 chars
Scraping: 3-branch... 1,907 chars
Scraping: 6-fall-back-topics... 3,021 chars
Scraping: custom-entities... 5,265 chars
Scraping: use-entities... 4,645 chars
Scraping: variables... 5,151 chars
Scraping: 2-power-automate... 6,843 chars
Scraping: actions... 2,996 chars
Scraping: triggers... 3,247 chars
Scraping: design... 6,669 chars
Scraping: conversational... 12,507 chars
Scraping: analytics... 3,939 chars
Scraping: bring-autonomous-agents-into-enterprise... 2,798 chars
Scraping: test-refine-autonomous-agent... 1,464 chars
Scraping: security-and-governance... 6,437 chars
Scra

## 3. Chunking



Each chunk carries metadata identifying its track (`audience`) and subject area (`domain`).

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
)

all_chunks = []
for doc in scraped_docs:
    chunks = text_splitter.split_text(doc["content"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(
            Document(
                page_content=chunk,
                metadata={
                    "source": doc["url"],
                    "slug": doc["slug"],
                    "domain": doc["domain"],
                    "audience": doc["audience"],
                    "chunk_index": i,
                },
            )
        )

audience_counts = Counter(c.metadata["audience"] for c in all_chunks)
domain_counts = Counter(c.metadata["domain"] for c in all_chunks)

print(f"Total chunks: {len(all_chunks)}")
print(f"\nBy track: {dict(audience_counts)}")
print(f"By domain: {dict(domain_counts)}")

Total chunks: 386

By track: {'ai_champion': 239, 'user': 147}
By domain: {'architecture': 207, 'connectors': 17, 'security': 12, 'governance': 3, 'overview': 36, 'adoption': 7, 'productivity': 42, 'communication': 32, 'prompting': 14, 'data-analysis': 16}


## 4. Embed and Upsert to Qdrant

We use OpenAI `text-embedding-3-small` (1536-dim vectors) and store in Qdrant Cloud.
`force_recreate=True` replaces any existing collection with fresh data.

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"Embedding and upserting {len(all_chunks)} chunks to '{COLLECTION_NAME}'...")

vector_store = QdrantVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name=COLLECTION_NAME,
    force_recreate=True,
)

print(f"Done! Collection '{COLLECTION_NAME}' created with {len(all_chunks)} points.")

Embedding and upserting 386 chunks to 'certops_docs'...
Done! Collection 'certops_docs' created with 386 points.


In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import PayloadSchemaType

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

for field in ["metadata.domain", "metadata.audience"]:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=PayloadSchemaType.KEYWORD,
    )
print("Payload indexes created for domain and audience.")

Payload indexes created for domain and audience.


## 5. Sanity Check

In [8]:
print(f"Collection '{COLLECTION_NAME}': {client.get_collection(COLLECTION_NAME).points_count} points")

results = vector_store.similarity_search("How do I create an agent in Copilot Studio?", k=3)
for i, doc in enumerate(results):
    print(f"  [{i+1}] audience={doc.metadata['audience']}, domain={doc.metadata['domain']}")
    print(f"      {doc.page_content[:120]}...")

Collection 'certops_docs': 386 points
  [1] audience=ai_champion, domain=architecture
      ### Agent creation using natural language

When you create an agent, you can describe what you want your agent to do in ...
  [2] audience=ai_champion, domain=architecture
      ### Copy a declarative agent to Copilot Studio

If you start building an agent in Microsoft 365 Copilot and want to add ...
  [3] audience=ai_champion, domain=architecture
      ## How to create an agent

Agents can be created in multiple ways. You can use the conversational experience to describe...


### Qdrant Dashboard — Sample Point

The screenshot below shows a single point in the `certops_docs` collection. Each point stores the chunk text (`page_content`), metadata payload (`source`, `slug`, `domain`, `audience`, `chunk_index`), and a 1536-dimensional embedding vector.

![Qdrant certops_docs collection](../qdrant_cert_ops.png)